<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/Week4_CLUSTERING_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder \
    .appName("BDA_Project_Week4_Clustering") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = spark.read.parquet("/content/drive/MyDrive/Parquet/02_AgriTrade_ValueOnly.parquet")

In [4]:
# Filter for import-related elements
import_elements = [
    row.Element for row in df.select("Element").distinct().collect()
    if row.Element is not None and "import" in row.Element.lower()
]

import_element = import_elements[0] if len(import_elements) > 0 else None
if import_element:
    df_trade = df.filter(F.col("Element") == import_element)
else:
    df_trade = df

print(f"Total records after filtering for imports: {df_trade.count():,}")

Total records after filtering for imports: 2,584,844


In [5]:
# Setup Window specification for YoY calculation
window_spec = Window.partitionBy("Area", "Item").orderBy("Year")

df_features = df_trade.withColumn(
    "Value_prev", F.lag("Value").over(window_spec)
)

df_features = df_features.withColumn(
    "YoY_Change",
    F.when(
        F.col("Value_prev").isNull() | (F.col("Value_prev") == 0), 0
    ).otherwise(
        ((F.col("Value") - F.col("Value_prev")) / F.col("Value_prev")) * 100
    )
).na.fill(0)

In [6]:
# Aggregate features by Area and Year
area_year_features = df_features.groupBy("Area", "Year").agg(
    F.countDistinct("Item").alias("num_products"),
    F.sum("Value").alias("total_volume"),
    F.avg("YoY_Change").alias("avg_growth")
).na.fill(0).filter(F.col("total_volume") > 0)

print(f"Final records available for clustering: {area_year_features.count():,}")
area_year_features.show(5)

Final records available for clustering: 11,904
+-------------------+----+------------+------------+------------------+
|               Area|Year|num_products|total_volume|        avg_growth|
+-------------------+----+------------+------------+------------------+
|             Angola|1979|          78|   2731547.0| 33.50274490225309|
|             Angola|1987|          99|   2822339.0| 80.46926497028974|
|Antigua and Barbuda|1995|         107|    516409.0|20.648089477221376|
|Antigua and Barbuda|2017|         314|  1259977.03| 70.56242981042807|
|          Argentina|2010|         292| 6.4724423E7| 843.8836402500788|
+-------------------+----+------------+------------+------------------+
only showing top 5 rows


In [7]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Step 1: Assemble features into a single Vector
feature_cols = ["num_products", "total_volume", "avg_growth"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_unscaled"
)

df_vector = assembler.transform(area_year_features)

In [8]:
# Step 2: Apply StandardScaler
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

print("Feature scaling is complete! The dataset is now mathematically ready for distance-based clustering.")

Feature scaling is complete! The dataset is now mathematically ready for distance-based clustering.


In [9]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

In [10]:
# Initialize tracking variables
best_k = 2
best_score = -1.0
k_values = [2, 3, 4, 5, 6, 7, 8]
scores = []

print("=== STARTING KMEANS HYPERPARAMETER TUNING ===")
print("Evaluating different values of 'k'...\n")

for k in k_values:
    # 1. Initialize KMeans with the current k
    kmeans_temp = KMeans(k=k, featuresCol="features", predictionCol="prediction", seed=42)

    # 2. Train the model
    model_temp = kmeans_temp.fit(df_scaled)

    # 3. Generate predictions
    predictions_temp = model_temp.transform(df_scaled)

    # 4. Calculate silhouette score
    score = evaluator.evaluate(predictions_temp)
    scores.append(score)

    print(f"[*] Tested Model (k={k}) -> Silhouette Score: {score:.4f}")

    # 5. Track the best model
    if score > best_score:
        best_score = score
        best_k = k

=== STARTING KMEANS HYPERPARAMETER TUNING ===
Evaluating different values of 'k' based on Lecture 10...

[*] Tested Model (k=2) -> Silhouette Score: 0.4976
[*] Tested Model (k=3) -> Silhouette Score: 0.5124
[*] Tested Model (k=4) -> Silhouette Score: 0.6788
[*] Tested Model (k=5) -> Silhouette Score: 0.7051
[*] Tested Model (k=6) -> Silhouette Score: 0.7634
[*] Tested Model (k=7) -> Silhouette Score: 0.6433
[*] Tested Model (k=8) -> Silhouette Score: 0.6462


In [11]:
print(f"\nTUNING COMPLETE")
print(f"The algorithm determined that the optimal number of clusters is: k = {best_k}")
print(f"This configuration achieved the highest Silhouette Score: {best_score:.4f}")


🏆 TUNING COMPLETE 🏆
The algorithm determined that the optimal number of clusters is: k = 6
This configuration achieved the highest Silhouette Score: 0.7634
